In [ ]:
import zipfile, os
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.model_selection import train_test_split

from transformers import BertConfig, BertModel, BertTokenizerFast, BertForSequenceClassification, TrainingArguments, Trainer, get_linear_schedule_with_warmup

In [ ]:
torch.cuda.empty_cache()

In [ ]:
path = kagglehub.dataset_download("senylar/sis-text-class")

print("Path to dataset files:", path)

In [ ]:
RANDOM_SEED = 1
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
data_train = pd.read_csv('/train.csv')
data_val = pd.read_csv('/valid.csv')

In [ ]:
data_train = data_train[0:10000]

Counter({1: 4856, 2: 2534, 0: 2610})

In [ ]:
data_val = data_val[0:2000]
Counter(data_val['sentiment'])

Counter({1: 928, 0: 543, 2: 529})

In [ ]:
def slice_token(index, sentences, labels, tokenizer, max_length):
    start, stop, step = index.indices(len(sentences))
    result = []
    for i in range(start, stop, step):
        encoding = tokenizer(
                [sentences[i]],
                padding='max_length',
                truncation = True,
                max_length = max_length,
                return_tensors = 'pt'
            )
        item = {key: val.squeeze(0) for key, val in encoding.items()}  # Убираем batch dim
        item['labels'] = torch.tensor(labels[i], dtype=torch.long) # Без лишнего .unsqueeze(1)
        result.append(item)
    return result


In [ ]:
class NERDataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_length):
        self.sentences = sentences.tolist()
        self.labels = labels.astype(float).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):

        if isinstance(idx, slice):
            # Обработка среза
            return slice_token(idx, self.sentences, self.labels, self.tokenizer, self.max_length)
        elif isinstance(idx, int):
            tokens = self.sentences[idx]
            tag = self.labels[idx]
            # токенизируем
            encoding = self.tokenizer(
                [tokens],
                padding='max_length',
                truncation = True,
                max_length = self.max_length,
                return_tensors = 'pt'
            )
            item = {key: val.squeeze(0) for key, val in encoding.items()}  # Убираем batch dim
            item['labels'] = torch.tensor(tag, dtype=torch.long) # Без лишнего

            return item

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('google-bert/bert-base-uncased')

In [ ]:
dataset_train = NERDataset(data_train['text'], data_train['sentiment'], tokenizer, 512)
#dataset_test = NERDataset(ds_test['comment'], ds_test['toxic'], tokenizer, 512)
dataset_val = NERDataset(data_val['text'], data_val['sentiment'], tokenizer, 512)

In [ ]:
train_loader = DataLoader(dataset_train[0:3000], batch_size=16)
val_loader = DataLoader(dataset_val[0:1000], batch_size = 16)

In [ ]:
test_loader = DataLoader(dataset_test, batch_size=16)
train_loader = DataLoader(dataset_train, batch_size=16)
val_loader = DataLoader(dataset_val, batch_size = 16)

In [ ]:
model = BertForSequenceClassification.from_pretrained('google-bert/bert-base-uncased', num_labels = 3)
model = model.to(device)

In [ ]:
#optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5, correct_bias=False)
training_args = TrainingArguments(
    output_dir='./results',          # Output directory
    eval_strategy="epoch",    # Evaluate after each epoch
    learning_rate=2e-5,             # Learning rate
    per_device_train_batch_size=16, # Batch size for training
    per_device_eval_batch_size=16,  # Batch size for evaluation
    num_train_epochs=3,             # Number of epochs
    weight_decay=0.01,              # Strength of weight decay
    logging_dir="./logs",           # Directory for storing logs
    logging_steps=10,               # что это такое?
    save_strategy="epoch",          # Save model after each epoch
    load_best_model_at_end=True,    # Load the best model after training
    metric_for_best_model="f1", # Use F1 score to choose the best model
    seed=RANDOM_SEED
)

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    # логиты в индексы
    predictions = predictions.argmax(axis=-1)

    # пихнем в метрику и получим результат
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    acc = accuracy_score(labels, predictions)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_loader.dataset,  # Training dataset
    eval_dataset=val_loader.dataset,   # Evaluation dataset
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
    )


In [ ]:
# собственно обучение - автоматически делает логи
trainer.train()

# оценим модельку
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

# Сохраним, что получилось
trainer.save_model("./ner_model")
#b5a31e3a762dc4fdbd905c7a205899ee8116917a

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.859000,0.919932,0.537000,0.468077,0.513363,0.474766
2,0.818400,0.876167,0.571000,0.539075,0.563004,0.533496
3,0.773300,0.876738,0.583000,0.553206,0.580352,0.545774


Evaluation Results: {'eval_loss': 0.8767378926277161, 'eval_accuracy': 0.583, 'eval_f1': 0.5532056057907663, 'eval_precision': 0.5803520553520554, 'eval_recall': 0.5457742268936957, 'eval_runtime': 29.98, 'eval_samples_per_second': 33.356, 'eval_steps_per_second': 2.101, 'epoch': 3.0}


In [ ]:
def eval_model(model, data_loader, device):
  model = model.eval()

  all_preds = torch.tensor([], device=device)
  all_trues = torch.tensor([], device=device)

  with torch.no_grad():
    for d in data_loader:
      input_ids = d["input_ids"].to(device)
      attention_mask = d["attention_mask"].to(device)
      targets = d["labels"].to(device)

      outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
      )
      #print(outputs)
      preds = torch.argmax(outputs['logits'], axis=-1)
      all_preds = torch.cat((all_preds, preds), -1)
      all_trues = torch.cat((all_trues, targets), -1)

  precision, recall, f1, _ = precision_recall_fscore_support(all_trues.cpu(), all_preds.cpu(), average='macro')
  acc = accuracy_score(all_trues.cpu(), all_preds.cpu())
  return {
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall
  }

In [ ]:
test = eval_model(model, test_loader, device)

In [ ]:
test

{'accuracy': 0.8848821081830791,
 'f1': 0.8657798352386528,
 'precision': 0.879727720472075,
 'recall': 0.8554682742662283}

In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(torch.tensor(data['toxic']), torch.zeros(14412), average='macro')
acc = accuracy_score(torch.tensor(data['toxic']), torch.zeros(14412))
print({
      'accuracy': acc,
      'f1': f1,
      'precision': precision,
      'recall': recall})

{'accuracy': 0.6651401609769636, 'f1': 0.3994499541628469, 'precision': 0.3325700804884818, 'recall': 0.5}


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
